<a href="https://colab.research.google.com/github/Rupali0/Advanced-Data-Science-Laboratory-/blob/main/Prompt_Based_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU:", torch.cuda.is_available())

GPU: True


In [2]:
!pip install -q "transformers<5" accelerate gradio pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small and fast; use 1.5B-Instruct if the GPU is on

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
if device == "cuda":
    model = model.half()
model = model.to(device).eval()
print("Model loaded on", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on cuda


In [ ]:
import torch, time, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- Load model ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model = (model.half() if device == "cuda" else model).to(device).eval()

def ask_llm(messages, max_new_tokens=150, temperature=0.2):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(device)
    gen = dict(max_new_tokens=max_new_tokens)
    gen.update(do_sample=True, temperature=temperature, top_p=0.9) if temperature > 0 else gen.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **gen)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---------- Prompts ----------
COLLEGE_SYSTEM = """You are a College Inquiry Assistant. Answer ONLY from the info below in 2-3 short sentences.
If not in the info, say: "I don't have that information. Please contact admissions@sunrise.edu."
INFO: Courses: B.Tech in Computer Engineering, AI & ML, Electronics, Mechanical (4 years).
Fees: B.Tech Rs 1,10,000/year, hostel Rs 60,000/year. Admissions: June to 15 July via MHT-CET/JEE.
Documents: 10th and 12th marksheets, CET/JEE scorecard, ID proof, photos. Placements: 85%, TCS, Infosys, Wipro.
Contact: admissions@sunrise.edu, 9 AM-5 PM Mon-Sat."""

HEALTH_SYSTEM = """You are a Healthcare Information Assistant. Give general health info only, in 3-4 simple sentences.
No diagnosis, no medicine dosage. For serious symptoms, advise seeing a doctor.
End with: "This is general information, not medical advice. Please consult a doctor." """

CLASSIFY_SYSTEM = """Classify the message as one word: COLLEGE (admissions, courses, fees, hostel, placements),
HEALTH (symptoms, illness, diet, sleep, stress, fitness) or OTHER. Reply with only that word."""

EMERGENCY_WORDS = ["chest pain", "cannot breathe", "can't breathe", "heavy bleeding", "unconscious", "stroke", "heart attack", "overdose"]
EMERGENCY_REPLY = "This may be an emergency. Call 112 (ambulance 108 in India) or go to the nearest hospital now."

# ---------- Workflow: safety check -> intent -> route -> log ----------
LOG = []
def chatbot(msg):
    start, t = time.time(), msg.lower()
    if any(w in t for w in EMERGENCY_WORDS):
        intent, reply = "EMERGENCY", EMERGENCY_REPLY
    else:
        label = ask_llm([{"role": "system", "content": CLASSIFY_SYSTEM}, {"role": "user", "content": msg}], 5, 0).upper()
        intent = next((l for l in ["COLLEGE", "HEALTH", "OTHER"] if l in label), "OTHER")
        if intent == "COLLEGE":
            reply = ask_llm([{"role": "system", "content": COLLEGE_SYSTEM}, {"role": "user", "content": msg}])
        elif intent == "HEALTH":
            reply = ask_llm([{"role": "system", "content": HEALTH_SYSTEM}, {"role": "user", "content": msg}], 200, 0.3)
        else:
            reply = "I can help with college admissions or general health guidance. Please ask about those."
    LOG.append({"question": msg, "intent": intent, "reply": reply, "seconds": round(time.time() - start, 2)})
    return reply

# ---------- Automated test ----------
for q in ["What is the B.Tech fee?", "Which documents are needed?", "Do you offer MBA?",
          "I have a headache", "How to sleep better?", "I have chest pain", "Who won the match?"]:
    print(f"Q: {q}\nA: {chatbot(q)}\n")

df = pd.DataFrame(LOG)
print(df[["question", "intent", "seconds"]])
print(df["intent"].value_counts())
df.to_csv("chatbot_log.csv", index=False)

# ---------- Live chat ----------
print("\nChat started. Type 'exit' to stop.")
while True:
    m = input("You: ")
    if m.lower() == "exit":
        break
    print("Bot:", chatbot(m), "\n")

Q: What is the B.Tech fee?
A: I can help with college admissions or general health guidance. Please ask about those.

Q: Which documents are needed?
A: To receive proper care and treatment for your condition, you will need the following:

1. **Medical Records**: Copies of all previous medical records, including any recent tests or treatments.
2. **Physical Examination Report**: A detailed report from your healthcare provider outlining your current health status and any issues noted during the examination.
3. **Diagnosis Report**: If available, this document provides a comprehensive summary of your medical history, symptoms, and potential diagnoses.

These documents are crucial for ensuring that your healthcare needs are met accurately and promptly.

Q: Do you offer MBA?
A: I can help with college admissions or general health guidance. Please ask about those.

Q: I have a headache
A: A headache can be caused by various factors such as stress, dehydration, or more serious conditions like